# Born Into Burden — Reproducible Analysis Notebook

## Objective
This notebook reproduces the analysis presented in the HTML report by rebuilding all figures directly from raw datasets.

## Research Question
Does low handwashing access correlate with higher dependency burden and lower life expectancy?

## Datasets Used
- UNICEF Indicator 1: Handwashing access
- UNICEF Indicator 2: Dependency ratio
- World Bank Metadata: Life expectancy

## Method Overview
- Filter to 2022 for cross-sectional analysis
- Merge datasets using ISO alpha-3 country codes
- Create matched dataset (103 countries)
- Generate 4 figures:
  1. World map
  2. Life expectancy comparison
  3. Dependency vs hygiene scatter
  4. Regional time trends


# Born Into Burden — Colab notebook rebuild

This notebook **recreates the analysis and charts directly from the CSV files** so it can be run in **Google Colab** without copying the HTML output.

## What this notebook does
- loads the 3 uploaded datasets
- rebuilds the 2022 matched comparison sample
- validates the report's headline numbers
- recreates the 4 report figures from data

## Important note
This notebook is designed to reproduce the **same analytical results and equivalent visuals** as the HTML preview. Minor pixel-level differences can still happen across environments because map geometry sources, fonts, package versions, and rendering engines can differ.


In [ ]:
# Colab setup
!pip -q install geopandas==0.14.4 shapely==2.0.4 pyogrio==0.9.0 matplotlib==3.8.4 pandas==2.2.2 numpy==1.26.4

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import geopandas as gpd

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["font.family"] = "DejaVu Serif"

DATA_DIR = Path(".")  # In Colab, upload the three CSVs into the working directory

IND1_PATH = DATA_DIR / "unicef_indicator_1.csv"
IND2_PATH = DATA_DIR / "unicef_indicator_2.csv"
META_PATH = DATA_DIR / "unicef_metadata.csv"

for p in [IND1_PATH, IND2_PATH, META_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p.name}. Upload it to the Colab session first.")

print("Files found:", IND1_PATH.name, IND2_PATH.name, META_PATH.name)

In [ ]:
# Load data
ind1_raw = pd.read_csv(IND1_PATH)
ind2_raw = pd.read_csv(IND2_PATH)
meta_raw = pd.read_csv(META_PATH)

print("indicator_1 shape:", ind1_raw.shape)
print("indicator_2 shape:", ind2_raw.shape)
print("metadata shape   :", meta_raw.shape)

In [ ]:
# Data preparation: recreate the matched 2022 sample used across Figures 1–3
q_order = ["Q1 (0-15%)", "Q2 (15-40%)", "Q3 (40-65%)", "Q4 (65-85%)", "Q5 (85-100%)"]

hw_all = (
    ind1_raw.loc[ind1_raw["sex"] == "Total", ["country", "alpha_3_code", "time_period", "obs_value"]]
    .rename(columns={"time_period": "year", "obs_value": "handwashing_pct"})
    .copy()
)

dep_2022 = (
    ind2_raw.loc[(ind2_raw["time_period"] == 2022) & (ind2_raw["sex"] == "Total"), ["alpha_3_code", "obs_value"]]
    .rename(columns={"obs_value": "dep_ratio"})
    .copy()
)

le_2022 = (
    meta_raw.loc[meta_raw["year"] == 2022, ["alpha_3_code", "Life expectancy at birth, total (years)"]]
    .rename(columns={"Life expectancy at birth, total (years)": "life_exp"})
    .dropna(subset=["life_exp"])
    .copy()
)

merged_2022 = (
    hw_all.loc[hw_all["year"] == 2022]
    .merge(dep_2022, on="alpha_3_code", how="inner")
    .merge(le_2022, on="alpha_3_code", how="inner")
    .copy()
)

bins = [-np.inf, 15.0, 40.0, 65.0, 85.0, np.inf]
merged_2022["hw_band"] = pd.cut(
    merged_2022["handwashing_pct"],
    bins=bins,
    labels=q_order,
    include_lowest=True,
    right=True,
)

print("Matched 2022 sample:", len(merged_2022), "countries")
merged_2022.head()

In [ ]:
# Validation against the report's headline values
summary = {}
summary["matched_2022_countries"] = int(len(merged_2022))

lowest_row = merged_2022.loc[merged_2022["handwashing_pct"].idxmin()]
summary["lowest_country"] = str(lowest_row["country"])
summary["lowest_handwashing_pct"] = float(lowest_row["handwashing_pct"])
summary["lowest_country_life_exp"] = float(lowest_row["life_exp"])

life_by_band = (
    merged_2022.groupby("hw_band", observed=False)["life_exp"]
    .mean()
    .reindex(q_order)
)

summary["life_expectancy_by_band"] = {k: float(v) for k, v in life_by_band.items()}
summary["life_expectancy_gap_q5_minus_q1"] = float(life_by_band.iloc[-1] - life_by_band.iloc[0])
summary["dependency_handwashing_correlation"] = float(
    merged_2022["dep_ratio"].corr(merged_2022["handwashing_pct"])
)

display(pd.DataFrame(
    {
        "metric": [
            "matched_2022_countries",
            "lowest_country",
            "lowest_handwashing_pct",
            "lowest_country_life_exp",
            "life_expectancy_gap_q5_minus_q1",
            "dependency_handwashing_correlation",
        ],
        "value": [
            summary["matched_2022_countries"],
            summary["lowest_country"],
            round(summary["lowest_handwashing_pct"], 3),
            round(summary["lowest_country_life_exp"], 3),
            round(summary["life_expectancy_gap_q5_minus_q1"], 3),
            round(summary["dependency_handwashing_correlation"], 6),
        ],
    }
))

assert summary["matched_2022_countries"] == 103
assert summary["lowest_country"] == "Liberia"
assert round(summary["lowest_handwashing_pct"], 1) == 3.4
assert round(summary["dependency_handwashing_correlation"], 2) == -0.77

print("Validation checks passed.")

In [ ]:
# Region helper logic for scatterplot and time series
def assign_region_from_iso(iso):
    south_asia = {"AFG", "BGD", "BTN", "IND", "LKA", "MDV", "NPL", "PAK"}
    mena = {
        "DZA", "BHR", "DJI", "EGY", "IRN", "IRQ", "ISR", "JOR", "KWT", "LBN",
        "LBY", "MAR", "OMN", "PSE", "QAT", "SAU", "SYR", "TUN", "ARE", "YEM"
    }
    europe_central_asia_extra = {
        "ARM", "AZE", "GEO", "KAZ", "KGZ", "TJK", "TKM", "UZB", "TUR"
    }
    north_america = {"USA", "CAN"}
    world_continent_map = world_gdf.set_index("iso_a3")["continent"].to_dict()

    if iso in south_asia:
        return "South Asia"
    if iso in mena:
        return "Middle East & North Africa"
    if iso in europe_central_asia_extra:
        return "Europe & Central Asia"
    if iso in north_america:
        return "North America"

    continent = world_continent_map.get(iso, None)
    if continent == "Africa":
        return "Sub-Saharan Africa"
    if continent == "Europe":
        return "Europe & Central Asia"
    if continent == "Asia":
        return "East Asia & Pacific"
    if continent == "South America":
        return "Latin America & Caribbean"
    if continent == "North America":
        return "Latin America & Caribbean"
    if continent == "Oceania":
        return "East Asia & Pacific"
    return "Other"

MAP_FILL = {
    "Q1 (0-15%)": "#d73027",
    "Q2 (15-40%)": "#fc8d59",
    "Q3 (40-65%)": "#fee090",
    "Q4 (65-85%)": "#91cf60",
    "Q5 (85-100%)": "#1a9850",
    "No data": "#cccccc",
}

REGION_COLOURS = {
    "Sub-Saharan Africa": "#c44e52",
    "South Asia": "#dd8452",
    "Middle East & North Africa": "#55a868",
    "Europe & Central Asia": "#4c72b0",
    "East Asia & Pacific": "#8172b2",
    "Latin America & Caribbean": "#937860",
    "North America": "#64b5cd",
    "Other": "#999999",
}

In [ ]:
# Load world geometry
try:
    world_gdf = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
except Exception:
    world_gdf = gpd.read_file(
        "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    )

world_gdf = world_gdf[world_gdf["name"] != "Antarctica"].copy()
world_gdf["region"] = world_gdf["iso_a3"].apply(assign_region_from_iso)

merged_2022["region"] = merged_2022["alpha_3_code"].apply(assign_region_from_iso)

print(sorted(merged_2022["region"].dropna().unique()))

In [ ]:
# Figure 1 — World map of handwashing access
world_map = world_gdf.merge(
    merged_2022[["alpha_3_code", "handwashing_pct", "hw_band"]],
    left_on="iso_a3",
    right_on="alpha_3_code",
    how="left",
)

world_map["fill_group"] = world_map["hw_band"].astype(object).fillna("No data")

fig, ax = plt.subplots(figsize=(14, 7))
world_map.plot(
    ax=ax,
    color=world_map["fill_group"].map(MAP_FILL),
    edgecolor="white",
    linewidth=0.25,
)

ax.set_title("Figure 1. Global Handwashing Access, 2022", pad=14)
ax.set_axis_off()

legend_handles = [Patch(facecolor=MAP_FILL[k], edgecolor="none", label=k) for k in MAP_FILL]
ax.legend(
    handles=legend_handles,
    title="Access level",
    loc="lower left",
    bbox_to_anchor=(0.01, 0.02),
    frameon=True,
)

plt.tight_layout()
plt.show()

In [ ]:
# Figure 2 — Average life expectancy by handwashing-access band
qs = (
    merged_2022.groupby("hw_band", observed=False)["life_exp"]
    .mean()
    .reindex(q_order)
    .reset_index()
)
qs.columns = ["hw_band", "avg_life_exp"]
qs["bar_label"] = qs["avg_life_exp"].round(1).astype(str) + " yrs"

global_avg = float(merged_2022["life_exp"].mean())

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(qs["hw_band"], qs["avg_life_exp"], edgecolor="black")

ax.axhline(global_avg, linestyle="--", linewidth=1.25)
ax.set_title("Figure 2. Average Life Expectancy by Handwashing Access Band (2022)")
ax.set_xlabel("Handwashing access band")
ax.set_ylabel("Life expectancy at birth (years)")
ax.set_ylim(0, max(qs["avg_life_exp"].max() + 10, 80))

for rect, label in zip(bars, qs["bar_label"]):
    ax.text(
        rect.get_x() + rect.get_width() / 2,
        rect.get_height() + 0.6,
        label,
        ha="center",
        va="bottom",
        fontsize=10,
    )

ax.text(
    len(qs) - 0.2,
    global_avg + 0.5,
    f"Dataset average = {global_avg:.1f}",
    ha="right",
    va="bottom",
    fontsize=10,
)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(qs)

In [ ]:
# Figure 3 — Dependency–hygiene scatterplot with regression
scatter_df = merged_2022.copy()
corr_r = scatter_df["dep_ratio"].corr(scatter_df["handwashing_pct"])

label_countries = ["Liberia", "Qatar", "Nigeria", "India", "Bangladesh", "Ethiopia"]
labels_df = scatter_df[scatter_df["country"].isin(label_countries)].copy()

nudge = {
    "Liberia": (1.5, 4),
    "Qatar": (-5, -5),
    "Nigeria": (1.5, 3),
    "India": (1.5, -5),
    "Bangladesh": (1.5, 3),
    "Ethiopia": (1.5, 3),
}

x = scatter_df["dep_ratio"].to_numpy()
y = scatter_df["handwashing_pct"].to_numpy()
m, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 200)
y_line = m * x_line + b

fig, ax = plt.subplots(figsize=(10, 6))

for region, group in scatter_df.groupby("region"):
    ax.scatter(
        group["dep_ratio"],
        group["handwashing_pct"],
        s=45,
        alpha=0.72,
        label=region,
        color=REGION_COLOURS.get(region, "#999999"),
    )

ax.plot(x_line, y_line, linestyle="--", linewidth=1.5, color="black")

for _, row in labels_df.iterrows():
    dx, dy = nudge.get(row["country"], (1.5, 0))
    ax.text(
        row["dep_ratio"] + dx,
        row["handwashing_pct"] + dy,
        row["country"],
        fontsize=9,
        style="italic",
    )

ax.set_title(f"Figure 3. The Dependency–Hygiene Nexus (r = {corr_r:.2f})")
ax.set_xlabel("Dependency ratio")
ax.set_ylabel("Handwashing access (%)")
ax.legend(title="Region", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 4 — Progress in the most affected region
# Rebuild a time series using yearly country-level handwashing data,
# then compare Sub-Saharan Africa with the full global sample.

hw_ts = hw_all.copy()
hw_ts["region"] = hw_ts["alpha_3_code"].apply(assign_region_from_iso)

region_year = (
    hw_ts.groupby(["region", "year"], as_index=False)["handwashing_pct"]
    .mean()
)

world_year = (
    hw_ts.groupby("year", as_index=False)["handwashing_pct"]
    .mean()
    .assign(region="World")
)

trend_df = pd.concat(
    [
        region_year.loc[region_year["region"] == "Sub-Saharan Africa"],
        world_year,
    ],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(10, 5.5))

for region_name, df_plot in trend_df.groupby("region"):
    ax.plot(
        df_plot["year"],
        df_plot["handwashing_pct"],
        linewidth=2.2,
        label=region_name,
    )

latest = trend_df.sort_values("year").groupby("region").tail(1)
for _, row in latest.iterrows():
    ax.text(row["year"] + 0.2, row["handwashing_pct"], row["region"], fontsize=10, va="center")

ax.set_title("Figure 4. Progress Not Fast Enough")
ax.set_xlabel("Year")
ax.set_ylabel("Average handwashing access (%)")
ax.set_xlim(hw_ts["year"].min(), hw_ts["year"].max() + 1)
ax.legend().remove()
plt.tight_layout()
plt.show()

display(
    trend_df.sort_values(["region", "year"]).groupby("region").tail(5).reset_index(drop=True)
)

In [ ]:
# Optional: save the figures if you want separate image files
# Re-run the plotting cells and add plt.savefig("filename.png", bbox_inches="tight") before plt.show().

print("Notebook complete. The figures above are recreated directly from the CSV files.")

## Key Findings

- Countries with low handwashing access consistently show:
  - Higher dependency ratios
  - Lower life expectancy
- The correlation between dependency burden and hygiene access is strongly negative (~ -0.77)
- A gap of ~10 years exists between lowest and highest access groups
- Progress remains slow in Sub-Saharan Africa

## Conclusion
The results confirm that access to basic hygiene infrastructure is strongly associated with broader demographic and health inequalities.

## Reproducibility Note
All figures in this notebook are generated directly from the CSV datasets. Running this notebook in Google Colab with the same files will reproduce the same analytical results.
